# 13. Nonparametric Bayes inference on manifolds

**Source span:** printed pp. 156-181; physical PDF pp. 171-196 in `Nonparametric Inference on Manifolds with Applications to Shape Spaces.pdf`.

This is original course prose and code built from the local source map. It uses the source for order, terminology, and concept coverage only; it does not copy textbook passages, exercise text, screenshots, page crops, or printed figures.

**Standalone goal.** Density estimation, posterior consistency intuition, and manifold mixture computation with spherical and SPD examples. The notebook is designed to be read without the PDF open: every major concept is paired with a generated visual, a computational representation, and a numeric sanity check.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Nonparametric-Inference-on-Manifolds-with-Applications-to-Shape-Spaces/part-03-bayes-on-manifolds/chapter-13-bayes-inference/13-bayes-inference.ipynb",
  "course_dir": "Nonparametric-Inference-on-Manifolds-with-Applications-to-Shape-Spaces",
  "course_title": "Nonparametric Inference on Manifolds: visualization-first course",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Nonparametric-Inference-on-Manifolds-with-Applications-to-Shape-Spaces/part-03-bayes-on-manifolds/chapter-13-bayes-inference/13-bayes-inference.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Nonparametric-Inference-on-Manifolds-with-Applications-to-Shape-Spaces/part-03-bayes-on-manifolds/chapter-13-bayes-inference/13-bayes-inference.ipynb",
  "notebook_title": "13. Nonparametric Bayes inference on manifolds",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/ml-geometry.txt",
  "runtime_profile": "ml_geometry"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Translation guide

| Source concept | Computational reading in this notebook |
| --- | --- |
| metric support | executable object, invariant, or check used to teach `metric support` without copying source text |
| kernel mixture | executable object, invariant, or check used to teach `kernel mixture` without copying source text |
| Dirichlet process | executable object, invariant, or check used to teach `Dirichlet process` without copying source text |
| posterior computation | executable object, invariant, or check used to teach `posterior computation` without copying source text |
| classification | executable object, invariant, or check used to teach `classification` without copying source text |

The library route is **pyriemann, matplotlib, scipy, numpy**. The choice is intentional: the course uses manifold and statistical-geometry packages when the object is curved or constrained, and uses plotting only to make that object inspectable.


## Concept route

**metric support.** In this notebook it is treated as a sample-space definition. The learner should be able to point to where it appears in the artifact, name the invariant it preserves, and say which later inference step would break if it were ignored.

**kernel mixture.** In this notebook it is treated as a computational representation. The learner should be able to point to where it appears in the artifact, name the invariant it preserves, and say which later inference step would break if it were ignored.

**Dirichlet process.** In this notebook it is treated as a statistical estimator. The learner should be able to point to where it appears in the artifact, name the invariant it preserves, and say which later inference step would break if it were ignored.

**posterior computation.** In this notebook it is treated as a uncertainty diagnostic. The learner should be able to point to where it appears in the artifact, name the invariant it preserves, and say which later inference step would break if it were ignored.

**classification.** In this notebook it is treated as a model-checking habit. The learner should be able to point to where it appears in the artifact, name the invariant it preserves, and say which later inference step would break if it were ignored.

The sequence is intentionally visual-first. First identify the sample space or quotient. Next identify the mean, projection, density, or tangent object that can be computed. Finally ask what invariant or residual certifies that the computation stayed on the intended manifold.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np


def find_book_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if candidate.name == 'Nonparametric-Inference-on-Manifolds-with-Applications-to-Shape-Spaces' and (candidate / 'AGENTS.md').exists():
            return candidate
    raise RuntimeError('Could not locate course root')

BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import ARTIFACT_ROOT, display_artifact, load_json, require_artifacts

CHAPTER_ID = 'chapter-13'
checks = load_json(ARTIFACT_ROOT / CHAPTER_ID / 'checks' / 'numeric-checks.json')
checks


## Visual 1: manifold-density-mixture.png

This artifact introduces the chapter's main geometric object. Inspect the labels, the plotted representative, and the invariant that remains after the relevant metric, embedding, quotient, or prior has been chosen. The visual is generated from synthetic data or exact constructions, so it teaches the role of the concept rather than reproducing any source figure.


In [ ]:
visual_1 = ARTIFACT_ROOT / CHAPTER_ID / 'figures' / 'manifold-density-mixture.png'
display_artifact(visual_1, width=820, height=540)


## Visual 2: spd-riemannian-mean.png

The second artifact is a check or companion view. Use it to answer a concrete question: does the projection return to the manifold, does the quotient identify the intended representatives, does uncertainty live in a valid tangent chart, or does the Bayesian object respect the sample-space support?


In [ ]:
visual_2 = ARTIFACT_ROOT / CHAPTER_ID / 'figures' / 'spd-riemannian-mean.png'
display_artifact(visual_2, width='100%', height=560)


## Applied lab

Change one modeling decision in the chapter route and predict the visual consequence before running new code. For this unit, start with **metric support** and ask what would happen if it were treated as ordinary Euclidean data. Then repeat the question for **classification**. This small exercise is the fastest way to see why the chapter's geometry is not decorative: the statistic changes when the invariance changes.

A good lab note for this notebook has three parts: the representative you changed, the invariant you expected to survive, and the check that would fail if the wrong geometry were used.


In [ ]:
probe = {
    'check_file_keys': sorted(checks.keys()),
    'numeric_values': {k: v for k, v in checks.items() if isinstance(v, (int, float, bool))},
}
assert probe['check_file_keys']
probe


## Takeaways

The source chapter's lesson can be read as a route from data type to geometry to inference. In this notebook, **metric support** defines the geometric stage, **kernel mixture** supplies the main computational representation, and **classification** is where the statistical check or modeling consequence becomes visible. Keep those roles separate when moving to the next chapter: confusing representative coordinates with quotient objects is the most common way to get a plausible but wrong answer.


In [ ]:
expected_artifacts = [
    ARTIFACT_ROOT / CHAPTER_ID / 'figures' / 'manifold-density-mixture.png',
    ARTIFACT_ROOT / CHAPTER_ID / 'figures' / 'spd-riemannian-mean.png',
]
artifact_sizes = require_artifacts(expected_artifacts)
final_checks = load_json(ARTIFACT_ROOT / CHAPTER_ID / 'checks' / 'final-sanity.json')
final_sanity = {
    'chapter': CHAPTER_ID,
    'artifact_count': len(artifact_sizes),
    'numeric_check_keys': sorted(checks.keys()),
    'source_pdf_pages': final_checks.get('source_pdf_pages'),
}
assert final_sanity['artifact_count'] == 2
assert final_sanity['source_pdf_pages'] == '171-196'
final_sanity
